In [1]:
import glob

import xarray as xr

da_instant = glob.glob("../data/raw/**/*instant.nc")
da_accum = glob.glob("../data/raw/**/*accum.nc")

print(da_accum)
print(da_instant)

['../data/raw/2006_1-5/data_stream-oper_stepType-accum.nc', '../data/raw/2012_6-10/data_stream-oper_stepType-accum.nc', '../data/raw/2000_1-5/data_stream-oper_stepType-accum.nc', '../data/raw/2000_11-12/data_stream-oper_stepType-accum.nc', '../data/raw/2000_6-10/data_stream-oper_stepType-accum.nc', '../data/raw/2001_1-5/data_stream-oper_stepType-accum.nc', '../data/raw/2001_11-12/data_stream-oper_stepType-accum.nc', '../data/raw/2001_6-10/data_stream-oper_stepType-accum.nc', '../data/raw/2002_1-5/data_stream-oper_stepType-accum.nc', '../data/raw/2002_11-12/data_stream-oper_stepType-accum.nc', '../data/raw/2002_6-10/data_stream-oper_stepType-accum.nc', '../data/raw/2003_1-5/data_stream-oper_stepType-accum.nc', '../data/raw/2003_11-12/data_stream-oper_stepType-accum.nc', '../data/raw/2003_6-10/data_stream-oper_stepType-accum.nc', '../data/raw/2004_1-5/data_stream-oper_stepType-accum.nc', '../data/raw/2004_11-12/data_stream-oper_stepType-accum.nc', '../data/raw/2004_6-10/data_stream-oper_

In [2]:
da_instant = xr.open_mfdataset(da_instant, chunks='auto')
da_accum = xr.open_mfdataset(da_accum, chunks='auto')

In [3]:
# checking experiment version (expver)
df_instant_expver = da_instant.expver.to_dataframe()
df_accum_expver = da_accum.expver.to_dataframe()

print(df_instant_expver.head())
print(df_accum_expver.head())

# check unqiue values for expver
df_instant_expver.expver.nunique()

                     number expver expver
valid_time                               
2000-01-01 00:00:00       0   0001   0001
2000-01-01 01:00:00       0   0001   0001
2000-01-01 02:00:00       0   0001   0001
2000-01-01 03:00:00       0   0001   0001
2000-01-01 04:00:00       0   0001   0001
                     number expver expver
valid_time                               
2000-01-01 00:00:00       0   0001   0001
2000-01-01 01:00:00       0   0001   0001
2000-01-01 02:00:00       0   0001   0001
2000-01-01 03:00:00       0   0001   0001
2000-01-01 04:00:00       0   0001   0001


expver    1
expver    1
dtype: int64

In [4]:
df_accum_expver.expver.nunique()

expver    1
expver    1
dtype: int64

Its safe to drop expver here, since the values are all `1`.

In [5]:
da_instant = da_instant.drop_vars("expver")
da_accum = da_accum.drop_vars("expver")

Transform `total precipitation`, as it is accumulated

In [6]:

tp_daily = da_accum.resample(valid_time="1D").last()

In [7]:
tp_daily

<xarray.Dataset> Size: 300MB
Dimensions:     (valid_time: 5114, latitude: 121, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 41kB 2000-01-01 ... 2013-12-31
  * latitude    (latitude) float64 968B 35.0 34.75 34.5 34.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 65.0 65.25 65.5 ... 94.5 94.75 95.0
    number      int64 8B 0
Data variables:
    tp          (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-22T11:53 GRIB to CDM+CF via cfgrib-0.9.1...

Resample instant variables also

In [8]:
instant_daily = da_instant.resample(valid_time="1D").mean()
instant_daily

<xarray.Dataset> Size: 1GB
Dimensions:     (valid_time: 5114, latitude: 121, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 41kB 2000-01-01 ... 2013-12-31
  * latitude    (latitude) float64 968B 35.0 34.75 34.5 34.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 65.0 65.25 65.5 ... 94.5 94.75 95.0
    number      int64 8B 0
Data variables:
    u10         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
    d2m         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-22T11:53 GRIB to CDM+CF via cfgrib-0.9.1...

In [9]:
ds_daily = xr.merge([
    instant_daily,
    tp_daily,
])

/tmp/ipykernel_42456/481290995.py:1: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  ds_daily = xr.merge([


In [10]:
ds_daily

<xarray.Dataset> Size: 1GB
Dimensions:     (valid_time: 5114, latitude: 121, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 41kB 2000-01-01 ... 2013-12-31
  * latitude    (latitude) float64 968B 35.0 34.75 34.5 34.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 65.0 65.25 65.5 ... 94.5 94.75 95.0
    number      int64 8B 0
Data variables:
    u10         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
    v10         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
    d2m         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
    tp          (valid_time, latitude, longitude) float32 299MB dask.array<chunksize=(101, 82, 82), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-03-22T11:53 GRIB to CDM+CF via cfgrib-0.9.1...

In [ ]:
import xarray as xr

ds = xr.open_dataset("../data/interim/combined_2020-2024_daily.nc", chunks="auto")
ds.dims

In [ ]:
ds.drop_vars("number")

In [ ]:
combined_df = ds.to_dataframe()

Using the above results we can create a merge script.
`src/scripts/merge.py`

In [ ]:
from constants import DATA_CONFIG
from ranges import year_range
from scripts import Operations

raw_dir: str =  "../" + DATA_CONFIG["paths"]["raw"]
export_dir: str = "../" + DATA_CONFIG["paths"]["interim"]

for year in year_range(2000, 2014):
    print(f"Start process for year: {year}")
    merged_dataset = (
        Operations(year=year, raw_dir=raw_dir)
        .deserialize_accum(drop_expver=True, drop_number=True)
        .deserialize_instant(drop_expver=True, drop_number=True)
        .resample_accum()
        .resample_instant()
        .merge()
        .build()
    )

    merged_dataset.serialize(dir=f"{export_dir}/combined_{year}.nc", complevel=9)